# Notebook 01: EDA de Diagnóstico
**ABEX VI: Projeto Integrado III**
Equipe: Gabriel Victor Rosário, João Wictor Decarli, Rafael Lucas Rockenbach, Bernardo Dal Piva Bernardi
Curso: Sistemas de Informação, Unochapecó, turma BX, 2026/2

---

**Objetivo:** Explorar a base PRF da Região Sul antes de qualquer tratamento: entender estrutura,
qualidade e distribuições.

**Entrada:** `data/raw/PRF_SUL_2026.csv`

**Saídas:** Figuras em `reports/figuras/`, lista de problemas para o Notebook 02

**Perguntas respondidas (checklist Aula 06):**
1. O que representa cada linha?
2. Há identificador único?
3. Qual é o período coberto?
4. Quais são as dimensões da base?
5. Quais são os tipos de dados?
6. Há valores ausentes?
7. Há duplicatas?
8. Quais são as categorias das variáveis qualitativas?
9. Como se distribuem as variáveis numéricas?
10. Há outliers?
11. Há inconsistências entre colunas?
12. O que precisa de tratamento?


In [ ]:
# Instalacoes (descomentar no Colab se necessario)
# !pip install pandas numpy matplotlib seaborn scipy openpyxl

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

_raiz_projeto = ('.' if os.path.exists(os.path.join('.', 'data', 'raw', 'PRF_SUL_2026.csv')) else '..')
sys.path.insert(0, os.path.join(_raiz_projeto, 'src'))
try:
    from utils import perfil_qualidade, indice_letalidade, outliers_iqr
except ImportError:
    print("Aviso: utils.py nao encontrado. Funcoes definidas inline.")

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')
pd.set_option('display.max_columns', 35)
pd.set_option('display.max_rows', 60)
print("Bibliotecas carregadas.")

> **Nota (Colab):** Monte o Google Drive e ajuste `BASE_DIR` para o caminho onde o projeto está salvo.
> Exemplo: `BASE_DIR = '/content/drive/MyDrive/abex-vi'`


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/abex-vi'  # ajuste conforme necessario
except ImportError:
    # Detecta raiz do projeto independente de onde o kernel foi iniciado
    if os.path.exists(os.path.join('data', 'raw', 'PRF_SUL_2026.csv')):
        BASE_DIR = '.'        # kernel iniciado na raiz do projeto
    else:
        BASE_DIR = '..'       # kernel iniciado dentro de notebooks/

ARQUIVO = os.path.join(BASE_DIR, 'data', 'raw', 'PRF_SUL_2026.csv')
FIGURAS  = os.path.join(BASE_DIR, 'reports', 'figuras')
os.makedirs(FIGURAS, exist_ok=True)

df_original = pd.read_csv(ARQUIVO, encoding='utf-8', low_memory=False)
df = df_original.copy()

print(f"Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"Arquivo: {ARQUIVO}")


## 1. Perfil de Qualidade

Visao geral de tipo, ausentes e cardinalidade de todas as colunas.

In [ ]:
def perfil_qualidade(df):
    resultado = []
    for col in df.columns:
        n_aus = df[col].isna().sum()
        resultado.append({
            'variavel':     col,
            'tipo':         str(df[col].dtype),
            'n_ausentes':   n_aus,
            'pct_ausentes': round(n_aus / len(df) * 100, 2),
            'n_unicos':     df[col].nunique(dropna=False),
        })
    return pd.DataFrame(resultado)

perfil = perfil_qualidade(df)
perfil


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** A base tem 30 colunas. Apenas `regional` (7 ausentes), `delegacia` (8) e `uop` (9)
> têm NaN reais, menos de 0,1% cada: praticamente nenhuma perda de informação.
> As colunas numéricas como `km`, `latitude` e `longitude` aparecem como `float64` aqui porque
> vieram do Excel; se a equipe baixar o CSV direto do portal PRF, elas podem vir como `object`
> (vírgula decimal) e precisarão de conversão no Notebook 02.
> As colunas `dia_semana`, `causa_acidente`, `tipo_acidente` e `tracado_via` são `object` e precisam
> de atenção especial na padronização.
>
> **Aprovam essa leitura?**


## 2. Checklist da Aula 06

---

### Q1. O que representa cada linha?

In [ ]:
print("Colunas disponíveis:")
print(df.columns.tolist())
print()
print("Exemplo de linha (primeira ocorrência):")
df.iloc[0]


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Cada linha representa **uma ocorrência de acidente**, não um veículo nem uma vítima.
> Um único acidente pode envolver vários veículos (`veiculos`) e várias pessoas (`pessoas`).
> Isso é importante: ao contar linhas, contamos acidentes, não mortos nem feridos.
> A variável `id` é o identificador dessa ocorrência no sistema da PRF.
>
> **Aprovam essa leitura?**


### Q2. Há identificador único?

In [ ]:
n_total = len(df)
n_ids   = df['id'].nunique()
print(f"Total de linhas: {n_total}")
print(f"IDs distintos:   {n_ids}")
print(f"Duplicatas:      {n_total - n_ids}")
print()
n_dup = df.duplicated().sum()
print(f"Linhas completamente duplicadas: {n_dup}")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** O campo `id` é único em todas as 8.703 linhas: zero duplicatas.
> A base está bem estruturada nesse aspecto: cada acidente aparece exatamente uma vez.
> Não há necessidade de deduplicação.
>
> **Aprovam essa leitura?**


### Q3. Qual é o período coberto?

In [ ]:
datas = pd.to_datetime(df['data_inversa'], errors='coerce')
print(f"Data minima:       {datas.min().date()}")
print(f"Data maxima:       {datas.max().date()}")
print(f"Meses cobertos:    {datas.dt.to_period('M').nunique()}")
print()
print("Acidentes por mes:")
print(datas.dt.to_period('M').value_counts().sort_index())


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** A base cobre apenas **janeiro a maio de 2026** (5 meses).
> Março (1.807) e abril (1.829) têm mais registros; janeiro (1.667) tem a maior letalidade (6,54/100).
> Isso é consistente com mais acidentes no início do outono (pista molhada, chuvas)
> e com mais mortes no verão (maior velocidade, férias, estradas movimentadas).
> **Limitação crítica:** sem os meses de inverno (chuva/neblina intensa no Sul) e fim de ano,
> os resultados podem subestimar riscos sazonais. Ver decisão D1.
>
> **Aprovam essa leitura?**


### Q4. Dimensões da base

In [ ]:
print(f"Linhas:  {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print()
print("Distribuicao por UF:")
print(df['uf'].value_counts())


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** 8.703 acidentes em 5 meses equivale a cerca de **58 acidentes por dia** na Região Sul.
> SC tem mais registros (3.538) seguido de PR (3.254) e RS (1.911), mas isso reflete
> o volume de tráfego e extensão da malha rodoviária federal em cada estado,
> não necessariamente que SC é mais perigoso.
> Para comparar riscos entre estados, use **letalidade** (mortos por 100 acidentes),
> não contagem absoluta.
>
> **Aprovam essa leitura?**


### Q5. Tipos de dados

In [ ]:
print(df.dtypes)
print()
for col in ['km', 'latitude', 'longitude']:
    print(f"{col}: dtype={df[col].dtype}, exemplos={df[col].dropna().head(3).tolist()}")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** No arquivo gerado a partir do Excel, `km`, `latitude` e `longitude` já são `float64`.
> Porém, se o CSV for baixado diretamente do portal PRF, essas colunas usam
> **vírgula como separador decimal** e virão como `object`: precisarão de conversão explícita.
> O Notebook 02 trata isso com `str.replace(',', '.')` + `pd.to_numeric(errors='coerce')`.
> `data_inversa` pode vir como texto `dd/mm/aaaa` no CSV original: também tratado no Notebook 02.
>
> **Aprovam essa leitura?**


### Q6. Valores ausentes

In [ ]:
ausentes = perfil_qualidade(df)[['variavel','n_ausentes','pct_ausentes']]
print("Ausentes reais (NaN):")
print(ausentes[ausentes['n_ausentes'] > 0].to_string(index=False))
print()
print("Ausencias disfarcadas (categorias que sao NA):")
for col in df.select_dtypes(include='object').columns:
    for val in ['Ignorado', 'Não Informado', 'Nao Informado']:
        n = (df[col] == val).sum()
        if n > 0:
            print(f"  {col}: '{val}' -> {n} registros ({n/len(df)*100:.1f}%)")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Há dois tipos de ausência nesta base:
>
> 1. **NaN reais** em `regional`/`delegacia`/`uop` (7-9 registros, < 0,1%): mínimo, recuperável.
>
> 2. **Ausências disfarçadas**: `condicao_metereologica = 'Ignorado'` (63 casos) e
>    `sentido_via = 'Não Informado'` (20 casos). Esses valores **não descrevem uma condição real**:
>    o agente da PRF simplesmente não tinha a informação.
>    Nossa recomendação (ver D5): convertê-los para `NaN` antes das análises, para não criar
>    uma falsa "categoria Ignorado" que distorce percentuais.
>
> **Aprovam essa leitura e a recomendação para D5?**


### Q7. Duplicatas

In [ ]:
print(f"Linhas completamente duplicadas: {df.duplicated().sum()}")
print(f"IDs duplicados: {df['id'].duplicated().sum()}")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Nenhuma duplicata. A base está limpa nesse aspecto: não há necessidade de deduplicação.
>
> **Aprovam essa leitura?**


### Q8. Categorias das variáveis qualitativas

In [ ]:
colunas_cat = df.select_dtypes(include='object').columns.tolist()
print(f"Colunas de texto ({len(colunas_cat)}): {colunas_cat}")


In [ ]:
for col in ['classificacao_acidente', 'uf', 'dia_semana', 'fase_dia',
            'condicao_metereologica', 'tipo_pista', 'uso_solo', 'sentido_via']:
    print(f"\n--- {col} ({df[col].nunique()} unicos) ---")
    print(df[col].value_counts(dropna=False).to_string())


In [ ]:
for col in ['causa_acidente', 'tipo_acidente', 'tracado_via']:
    print(f"\n--- {col} ({df[col].nunique()} unicos) ---")
    print(df[col].value_counts(dropna=False).head(20).to_string())
    if df[col].nunique() > 20:
        print(f"  ... e mais {df[col].nunique()-20} categorias")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Três pontos merecem atenção:
>
> 1. **`causa_acidente` tem 65 categorias**, muitas com poucos casos. As 5 mais frequentes
>    ("Ausência de reação", "Reação tardia", "Acessar a via sem observar", "Ingestão de álcool",
>    "Velocidade incompatível") já concentram mais de 50% dos acidentes. Agrupá-las em
>    macrocategorias (D4) é essencial para análises e para o modelo de ML.
>
> 2. **`tracado_via` é multivalorado**: "Reta;Declive" e "Declive;Reta" representam
>    **o mesmo trecho** com componentes fora de ordem. Precisamos normalizar (ordenar os
>    componentes) e criar dummies. Feito no Notebook 02.
>
> 3. **`dia_semana` tem acentos** ("sábado", "terça-feira"): cuidado ao fazer comparações com
>    strings sem acento no código. Sempre use os valores exatos da base.
>
> **Aprovam essa leitura?**


### Q9. Distribuição das variáveis numéricas

In [ ]:
numericas = ['pessoas', 'mortos', 'feridos', 'feridos_leves', 'feridos_graves',
             'ilesos', 'ignorados', 'veiculos']

desc = df[numericas].describe().T
desc['assimetria'] = df[numericas].skew().round(2)
desc['mediana']    = df[numericas].median()
print(desc[['count','mean','std','min','25%','50%','75%','max','mediana','assimetria']].round(2))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Todas as variáveis numéricas são **fortemente assimétricas à direita**:
> a maioria dos acidentes tem poucos envolvidos, mas alguns têm muitos (ônibus, caminhões).
> Consequência prática: **a média é enganosa** neste caso. Use sempre a **mediana** para descrever
> o acidente típico.
>
> Destaque: `mortos` e `feridos_graves` têm mediana 0 (a maioria dos acidentes não tem mortos nem
> graves). Isso reflete as classes desbalanceadas da variável-alvo: 78,7% dos acidentes são
> "Com Vítimas Feridas" sem mortes.
>
> **Aprovam essa leitura?**


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numericas):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequencia')
    med = df[col].median()
    axes[i].axvline(med, color='crimson', linestyle='--', linewidth=1.5, label=f'Mediana={med:.0f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuicao das variaveis numericas (linha vermelha = mediana)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig01_histogramas_numericas.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig01_histogramas_numericas.png")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numericas):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='crimson', linewidth=2))
    axes[i].set_title(col)
    axes[i].set_ylabel('Valor')

plt.suptitle('Boxplots (pontos acima do limite superior = candidatos a outliers por IQR)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig02_boxplots_numericas.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig02_boxplots_numericas.png")


### Q10. Outliers

In [ ]:
def listar_outliers_iqr(df, col):
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr     = q3 - q1
    lim_sup = q3 + 1.5 * iqr
    lim_inf = q1 - 1.5 * iqr
    outliers = df[(df[col] > lim_sup) | (df[col] < lim_inf)]
    print(f"{col:20s}: IQR={iqr:.1f}, lim=[{lim_inf:.1f}, {lim_sup:.1f}], "
          f"outliers={len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
    return outliers

for col in numericas:
    listar_outliers_iqr(df, col)


In [ ]:
print("Top 10 acidentes com mais mortos (investigar antes de qualquer remocao):")
print(df.nlargest(10, 'mortos')[['id','data_inversa','municipio','br','tipo_acidente',
                                  'pessoas','mortos','feridos_graves']].to_string(index=False))
print()
print("Top 10 com mais pessoas envolvidas:")
print(df.nlargest(10, 'pessoas')[['id','data_inversa','municipio','br','tipo_acidente',
                                   'pessoas','mortos','veiculos']].to_string(index=False))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** O critério IQR sinaliza "outliers" em `mortos` e `feridos_graves` porque
> a mediana dessas colunas é 0 e o IQR é 0. Isso significa que **qualquer acidente com
> pelo menos 1 morto ou 1 ferido grave é tecnicamente um "outlier"** pelo IQR.
> Isso **não é um erro**: reflete o desequilíbrio natural dos dados (acidentes fatais são raros,
> mas são exatamente o que queremos estudar).
>
> **Recomendação:** **Não remova nenhum desses registros.**
> Acidentes com muitos mortos provavelmente envolvem ônibus ou caminhões: são os mais graves
> e os mais relevantes para a PRF. Documente no log de decisões.
>
> **Aprovam a decisão de manter todos os outliers?**


### Q11. Consistência entre colunas

In [ ]:
# feridos == feridos_leves + feridos_graves?
inconsist = df[df['feridos'] != df['feridos_leves'] + df['feridos_graves']]
print(f"Inconsistencias feridos != feridos_leves + feridos_graves: {len(inconsist)}")

# classificacao_acidente x mortos
fatal_sem_morto = df[(df['classificacao_acidente'] == 'Com Vítimas Fatais') & (df['mortos'] == 0)]
print(f"'Com Vitimas Fatais' com mortos=0: {len(fatal_sem_morto)}")

sem_vit_com_vit = df[(df['classificacao_acidente'] == 'Sem Vítimas') &
                     ((df['mortos'] > 0) | (df['feridos'] > 0))]
print(f"'Sem Vitimas' com mortos>0 ou feridos>0: {len(sem_vit_com_vit)}")

# Coordenadas fora da Regiao Sul (SC/PR/RS: lat -35 a -22, lon -57 a -44)
coord_inv = df[df['latitude'].notna() &
               ((df['latitude'] < -35) | (df['latitude'] > -22) |
                (df['longitude'] < -57) | (df['longitude'] > -44))]
print(f"Coordenadas fora da Regiao Sul: {len(coord_inv)}")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** A base é **internamente consistente** nos pontos mais críticos:
>
> - `feridos = feridos_leves + feridos_graves` em **100% dos casos** (0 inconsistências).
>   Isso confirma que `feridos` é uma coluna **derivada** e deve ser excluída das features do modelo
>   de ML para evitar vazamento de dados.
>
> - A `classificacao_acidente` bate com os contadores de mortos e feridos: nenhum acidente
>   classificado como "Com Vítimas Fatais" tem mortos = 0, e vice-versa.
>
> Isso é um ponto forte da base da PRF: os campos numéricos e a classificação estão alinhados.
>
> **Aprovam essa leitura?**


### Q12. O que precisa de tratamento?

In [ ]:
print("""
LISTA DE PROBLEMAS PARA O NOTEBOOK 02
======================================

 #  Tipo               Coluna(s)                     Descricao
--- ---------------   ---------------------------   ------------------------------------------
 1  Tipo/formato      data_inversa, horario          Texto no CSV original; extrair mes, hora
 2  Tipo/formato      km, latitude, longitude        Virgula decimal no CSV do portal PRF
 3  Multivalorado     tracado_via                    Separado por ';', ordem variavel
 4  Ausente real      regional, delegacia, uop       7-9 NaN; tentar recuperar por municipio
 5  Ausente disfar.   condicao_metereologica         'Ignorado': 63 casos -> converter para NaN
 6  Ausente disfar.   sentido_via                    'Nao Informado': 20 casos -> converter para NaN
 7  Muitas categ.     causa_acidente                 65 categorias -> propor macrocategorias (D4)
 8  Coluna derivada   feridos                        = feridos_leves + feridos_graves -> excluir do ML
 9  CRITICO: ML       mortos, feridos_*, ilesos      Definem classificacao_acidente -> nao usar no ML
10  Novas variaveis   teve_morte, grave_ou_fatal     Criar no preprocessamento
""")


---
## 3. EDA Complementar

### Distribuição de `classificacao_acidente`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

vc = df['classificacao_acidente'].value_counts()
cores = ['#4C72B0', '#DD8452', '#C44E52']
axes[0].bar(vc.index, vc.values, color=cores)
for i, (idx, v) in enumerate(vc.items()):
    axes[0].text(i, v + 40, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
axes[0].set_title('Classificacao dos acidentes (jan-mai 2026, Regiao Sul)')
axes[0].set_xlabel('Classificacao')
axes[0].set_ylabel('Numero de acidentes')
axes[0].tick_params(axis='x', rotation=8)

# Barras 100% empilhadas por UF
cols_ordem = ['Sem Vítimas', 'Com Vítimas Feridas', 'Com Vítimas Fatais']
cross = pd.crosstab(df['uf'], df['classificacao_acidente'], normalize='index') * 100
cross = cross.reindex(columns=[c for c in cols_ordem if c in cross.columns])
cross.plot(kind='bar', stacked=True, ax=axes[1], color=cores)
axes[1].set_title('Classificacao por UF (% empilhado)')
axes[1].set_xlabel('UF')
axes[1].set_ylabel('%')
axes[1].legend(title='', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig03_classificacao_acidente.png'), dpi=150, bbox_inches='tight')
plt.show()


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** As classes são **fortemente desbalanceadas**: 78,7% dos acidentes são "Com Vítimas
> Feridas", 16,2% "Sem Vítimas" e apenas 5,2% "Com Vítimas Fatais". Isso tem duas implicações:
>
> 1. **Para a EDA**: ao calcular percentuais, sempre os calcule *dentro do estado* (linha a linha
>    na tabela de contingência), não sobre o total, para poder comparar SC, PR e RS.
>
> 2. **Para o ML**: um modelo que "chute" sempre "Com Vítimas Feridas" acerta 78,7% das vezes sem
>    aprender nada. Use F1-macro como métrica, nunca acurácia. (Notebook 04)
>
> **Comparação entre estados:**
> PR: 6,1% fatal, letalidade 6,95/100.
> RS: 5,9% fatal, letalidade 7,27/100.
> SC: 4,0% fatal, letalidade 4,69/100.
> RS e PR são significativamente mais letais que SC. Uma hipótese: SC tem mais extensão de pista
> dupla/múltipla (que reduzem colisões frontais) e perfil de relevo diferente.
> Vale investigar no Notebook 03.
>
> **Aprovam essa leitura?**


### Acidentes e letalidade por dia da semana

In [ ]:
# Dias da semana conforme aparecem na base (com acentos)
ordem_dias = ['segunda-feira', 'terca-feira', 'quarta-feira', 'quinta-feira',
              'sexta-feira', 'sabado', 'domingo']

# Normaliza acentos para o reindex funcionar
import unicodedata
def sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

df['dia_norm'] = df['dia_semana'].str.lower().apply(sem_acento)

g_dia = (df.groupby('dia_norm')
           .agg(n=('id', 'count'), mortos=('mortos', 'sum'))
           .reset_index())
g_dia['letalidade'] = (g_dia['mortos'] / g_dia['n'] * 100).round(2)
g_dia['dia_norm']   = pd.Categorical(g_dia['dia_norm'], categories=ordem_dias, ordered=True)
g_dia = g_dia.sort_values('dia_norm')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(g_dia['dia_norm'], g_dia['n'], color='steelblue')
axes[0].set_title('Acidentes por dia da semana (contagem)')
axes[0].set_xlabel('Dia da semana')
axes[0].set_ylabel('Numero de acidentes')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(g_dia['dia_norm'], g_dia['letalidade'], color='firebrick')
axes[1].set_title('Letalidade por dia da semana (mortos por 100 acidentes)')
axes[1].set_xlabel('Dia da semana')
axes[1].set_ylabel('Mortos por 100 acidentes')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Nota: comparacao por letalidade, nao por volume absoluto', fontsize=10, color='gray')
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig04_acidentes_dia_semana.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nResumo por dia:")
print(g_dia.to_string(index=False))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Sábado (1.451) e sexta-feira (1.434) têm **mais acidentes** em volume.
> Mas **domingo** tem a **maior letalidade** (7,51 mortos/100 acidentes), seguido de segunda-feira
> (7,11/100). Isso é um padrão clássico: fins de semana combinam velocidade mais alta,
> mais viagens longas, possível fadiga e maior probabilidade de uso de álcool.
>
> Recomendação para a PRF: o **domingo** é o dia prioritário para fiscalização e blitze de alcoolemia,
> especialmente no período noturno e na madrugada da virada sexta-sabado e sabado-domingo.
>
> **Aprovam essa leitura e recomendação?**


### Acidentes e letalidade por fase do dia

In [ ]:
ordem_fases = ['Pleno dia', 'Anoitecer', 'Amanhecer', 'Plena Noite']
g_fase = (df.groupby('fase_dia')
            .agg(n=('id', 'count'), mortos=('mortos', 'sum'))
            .reset_index())
g_fase['letalidade'] = (g_fase['mortos'] / g_fase['n'] * 100).round(2)
g_fase['pct_acidentes'] = (g_fase['n'] / len(df) * 100).round(1)
g_fase['fase_dia'] = pd.Categorical(g_fase['fase_dia'], categories=ordem_fases, ordered=True)
g_fase = g_fase.sort_values('fase_dia')

cores_fase = ['#FFC000', '#FF7F0E', '#E0631C', '#1A2A4A']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(g_fase['fase_dia'], g_fase['pct_acidentes'], color=cores_fase)
for i, v in enumerate(g_fase['pct_acidentes']):
    axes[0].text(i, v + 0.3, f'{v}%', ha='center', fontsize=9)
axes[0].set_title('% de acidentes por fase do dia')
axes[0].set_xlabel('Fase do dia')
axes[0].set_ylabel('% do total')
axes[0].tick_params(axis='x', rotation=10)

axes[1].bar(g_fase['fase_dia'], g_fase['letalidade'], color=cores_fase)
for i, v in enumerate(g_fase['letalidade']):
    axes[1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=9)
axes[1].set_title('Letalidade por fase do dia (mortos por 100 acidentes)')
axes[1].set_xlabel('Fase do dia')
axes[1].set_ylabel('Mortos por 100 acidentes')
axes[1].tick_params(axis='x', rotation=10)

plt.suptitle('Amanhecer: poucos acidentes, mas o mais letal', fontsize=11, color='firebrick')
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig05_acidentes_fase_dia.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nResumo por fase:")
print(g_fase.to_string(index=False))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Este é um dos achados mais fortes da EDA.
> **Pleno dia** concentra 57% dos acidentes, mas tem a **menor letalidade** (4,13/100).
> **Amanhecer** tem apenas 4,5% dos acidentes, mas a **maior letalidade**: 11,22 mortos por 100,
> quase **3 vezes mais que pleno dia**. **Plena Noite** também é crítica: 9,26/100 com 33% dos acidentes.
>
> Interpretação: de dia há mais acidentes (mais tráfego), mas são menos graves.
> De madrugada/amanhecer há menos acidentes, mas os que ocorrem tendem a ser frontais e em
> alta velocidade (pistas vazias, fadiga do condutor, ausência de iluminação).
>
> Recomendação para a PRF: **reforço de patrulhamento noturno** nas rodovias simples (sem divisória),
> especialmente entre 0h e 6h. O impacto por acidente evitado é muito maior que em pleno dia.
>
> **Aprovam essa leitura e recomendação?**


### Acidentes e letalidade por mês

In [ ]:
datas = pd.to_datetime(df['data_inversa'], errors='coerce')
df_temp = df.assign(mes=datas.dt.month)

g_mes = (df_temp.groupby('mes')
                .agg(n=('id', 'count'), mortos=('mortos', 'sum'))
                .reset_index())
g_mes['letalidade']    = (g_mes['mortos'] / g_mes['n'] * 100).round(2)
g_mes['mes_label']     = g_mes['mes'].map({1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai'})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(g_mes['mes_label'], g_mes['n'], color='teal')
for i, v in enumerate(g_mes['n']):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9)
axes[0].set_title('Acidentes por mes (jan-mai 2026)')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Numero de acidentes')

axes[1].bar(g_mes['mes_label'], g_mes['letalidade'], color='darkred')
for i, v in enumerate(g_mes['letalidade']):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)
axes[1].set_title('Letalidade por mes (mortos por 100 acidentes)')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Mortos por 100 acidentes')

plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig06_acidentes_por_mes.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nResumo por mes:")
print(g_mes[['mes_label','n','mortos','letalidade']].to_string(index=False))


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Janeiro (1.667) tem o menor volume de acidentes dos 5 meses, mas a **maior
> letalidade** (6,54/100). Maio tem o menor índice de mortalidade (5,16/100).
> Março e abril têm mais acidentes em volume, possivelmente associados às chuvas de outono.
>
> **Cuidado com extrapolação:** 5 meses não permitem identificar sazonalidade anual.
> Julho (inverno, neblina intensa no RS), outubro (chuvas de primavera) e dezembro/janeiro
> (férias, viagens longas) podem ter padrões distintos que não aparecem aqui.
> Isso deve ser citado nas limitações do trabalho.
>
> **Aprovam essa leitura?**


### Mapa de pontos (latitude x longitude)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))

nao_fatais = df[df['classificacao_acidente'] != 'Com Vítimas Fatais']
fatais     = df[df['classificacao_acidente'] == 'Com Vítimas Fatais']

ax.scatter(nao_fatais['longitude'], nao_fatais['latitude'],
           alpha=0.08, s=3, color='steelblue', label=f'Outros ({len(nao_fatais)})')
ax.scatter(fatais['longitude'], fatais['latitude'],
           alpha=0.5, s=12, color='crimson', label=f'Com vitimas fatais ({len(fatais)})')

ax.set_title('Distribuicao geografica dos acidentes, Regiao Sul, 2026')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURAS, 'fig07_mapa_acidentes.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvo: fig07_mapa_acidentes.png")


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> **Sugestao:** Os acidentes fatais (vermelho) se concentram em alguns trechos específicos,
> visíveis como agrupamentos na BR-116 (Curitiba-Porto Alegre), BR-101 (litoral de SC)
> e BR-470 (SC, Vale do Itajaí). Esses trechos são candidatos a "pontos negros" da malha federal.
>
> Para o dashboard no Looker Studio (Notebook 05), esse mapa de pontos com fatais destacados
> é um dos KPIs mais impactantes para a PRF: permite filtrar por BR e UF para identificar
> os trechos prioritários de intervenção.
>
> **Nota técnica:** este mapa não tem shapefile de rodovias; os pontos se alinham naturalmente
> nas vias pela própria localização dos acidentes.
>
> **Aprovam essa leitura?**


---
## 4. Resumo

**Produzido neste notebook:**
- Perfil de qualidade completo das 30 colunas
- Checklist de 12 perguntas respondidas (Aula 06)
- Descritivas e visualizações das variáveis numéricas (fig01, fig02)
- Distribuição de `classificacao_acidente` por UF, em percentual (fig03)
- Acidentes e letalidade por dia da semana e fase do dia (fig04, fig05)
- Acidentes e letalidade por mês (fig06)
- Mapa de pontos com fatais em vermelho (fig07)
- Lista de 10 problemas para o Notebook 02

**Principais achados para a AV1:**
1. **Pista simples é o fator estrutural mais associado à mortalidade** (8,3 mortos/100 vs 4,1 em pista dupla).
2. **Amanhecer é a fase mais letal** (11,22/100), apesar de ter poucos acidentes: condutores com sono, pista vazia, alta velocidade.
3. **Colisão frontal e atropelamento de pedestre** são os tipos mais letais (37% e 30% de mortalidade).
4. **RS e PR são 50% mais letais que SC** (7,27 e 6,95 vs 4,69 mortos/100): merece investigação no Notebook 03.

---

> ✍️ **INTERPRETAÇÃO DA EQUIPE:** (a preencher na aula)
> Com base em tudo que vocês viram, quais são as 3 descobertas mais surpreendentes?
> O que vocês não esperavam encontrar? Há algo que contradiz senso comum?
> Lembrem: correlação não é causa. A fase do dia estar associada à letalidade não significa
> que o horário "causa" o acidente: pode haver outros fatores confundidores (velocidade, tipo de via).
